# Classificar e reconhecer entidades em texto

Exercício complementar de 30 minutos.

**Parte 1, classificar:** um modelo clássico e um transformer disputam a mesma tarefa, com reviews reais.

**Parte 2, reconhecer:** um BERT pré-treinado marca nomes, empresas e locais no texto.

**Parte 3, mascarar:** o Banco Aurora, uma empresa fictícia, precisa esconder dados pessoais antes de enviar conversas a um LLM externo.

Antes de começar: **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4**.

In [ ]:
import json, os, re, sys, time, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
RAPIDO = os.environ.get("EX_RAPIDO") == "1"   # teste automatizado, com menos dados
REPO = "https://github.com/pgcferreira-stack/exercicios-complementares-fgv"

if IN_COLAB:
    if not os.path.exists("exercicios-complementares-fgv"):
        !git clone -q --depth 1 {REPO}
    os.chdir("exercicios-complementares-fgv/ner")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

transformers.logging.set_verbosity_error()
DEVICE = os.environ.get("EX_DEVICE") or ("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
pd.set_option("display.max_colwidth", 160)
print("Device:", DEVICE)

---
# Parte 1: classificar

## 1. Os dados

O B2W-Reviews01 reúne cerca de 130 mil avaliações reais de compras online. Notas 1 e 2 viram **negativo**, notas 4 e 5 viram **positivo**,
e a nota 3 fica de fora. Usamos uma amostra balanceada, com o mesmo número de reviews em cada categoria.

In [ ]:
URL = ("https://huggingface.co/datasets/ruanchaves/b2w-reviews01/resolve/"
       "refs%2Fconvert%2Fparquet/default/train/0000.parquet")
bruto = pd.read_parquet(URL, columns=["review_text", "overall_rating"]).dropna()
reviews = bruto[bruto.overall_rating != 3].copy()
reviews["label"] = (reviews.overall_rating >= 4).astype(int)

N_POR_CLASSE = 100 if RAPIDO else 1500
amostra = reviews.groupby("label").sample(N_POR_CLASSE, random_state=42)
treino, teste = train_test_split(amostra, test_size=0.2, stratify=amostra.label, random_state=42)
print(f"Base: {len(bruto):,} reviews | treino: {len(treino)} | teste: {len(teste)}")
amostra[["review_text", "overall_rating"]].sample(3, random_state=1)

## 2. Modelo clássico: TF-IDF e regressão logística

Cada review vira um vetor com o peso de cada palavra e de cada par de palavras. O modelo aprende um peso por termo.

In [ ]:
NOMES = ["negativo", "positivo"]

inicio = time.time()
classico = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                         LogisticRegression(max_iter=1000))
classico.fit(treino.review_text, treino.label)
tempo_classico = time.time() - inicio

pred_classico = classico.predict(teste.review_text)
print(f"Treino: {tempo_classico:.1f} s\n")
print(classification_report(teste.label, pred_classico, target_names=NOMES, digits=3))

**Como ler as métricas**

- **Precisão de "negativo":** das reviews marcadas como negativas, quantas eram mesmo negativas?
- **Recall de "negativo":** das reviews negativas, quantas o modelo encontrou?
- **F1:** resume as duas em um número.

## 3. Transformer: ajuste fino do BERTimbau

O BERTimbau é um BERT pré-treinado em português. Colocamos uma camada de classificação em cima e ajustamos por uma época.

In [ ]:
from datasets import Dataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          Trainer, TrainingArguments)

MODELO = "neuralmind/bert-base-portuguese-cased"
tok = AutoTokenizer.from_pretrained(MODELO)

def preparar(df):
    ds = Dataset.from_pandas(df[["review_text", "label"]], preserve_index=False)
    return ds.map(lambda lote: tok(lote["review_text"], truncation=True, max_length=128),
                  batched=True, remove_columns=["review_text"])

bert = AutoModelForSequenceClassification.from_pretrained(
    MODELO, num_labels=2, id2label=dict(enumerate(NOMES)), label2id={n: i for i, n in enumerate(NOMES)})
args = TrainingArguments(output_dir="saida", num_train_epochs=1, learning_rate=3e-5,
                         per_device_train_batch_size=32, per_device_eval_batch_size=64,
                         fp16=DEVICE == "cuda", use_cpu=DEVICE == "cpu",
                         logging_steps=20, save_strategy="no", report_to="none")
trainer = Trainer(model=bert, args=args, train_dataset=preparar(treino),
                  data_collator=DataCollatorWithPadding(tok))

inicio = time.time()
trainer.train()
tempo_bert = time.time() - inicio

pred_bert = trainer.predict(preparar(teste)).predictions.argmax(-1)
print(f"Treino: {tempo_bert:.0f} s\n")
print(classification_report(teste.label, pred_bert, target_names=NOMES, digits=3))

## 4. O transformer vale o custo?

Desempenho é só uma coluna da decisão.

In [ ]:
comparacao = pd.DataFrame({
    nome: {"Acurácia": round(accuracy_score(teste.label, p), 3),
           "F1": round(f1_score(teste.label, p, average="macro"), 3),
           "Recall negativo": round(recall_score(teste.label, p, pos_label=0), 3),
           "Treino (s)": round(t, 1),
           "Parâmetros": f"{n:,}"}
    for nome, p, t, n in [
        ("TF-IDF + regressão", pred_classico, tempo_classico, classico[-1].coef_.size + 1),
        ("BERTimbau", pred_bert, tempo_bert, bert.num_parameters())]
}).T
display(comparacao)

erros = teste.assign(classico=pred_classico, bert=pred_bert)
so_bert = erros[(erros.bert == erros.label) & (erros.classico != erros.label)]
so_classico = erros[(erros.classico == erros.label) & (erros.bert != erros.label)]
print(f"Só o BERT acertou: {len(so_bert)} | só o clássico acertou: {len(so_classico)}")
so_bert[["review_text", "overall_rating"]].head(5)

Agora teste frases difíceis. Troque as frases abaixo pelas suas.

In [ ]:
frases = [
    "Não é ruim, mas esperava mais pelo preço.",
    "Nada a reclamar, chegou antes do prazo!",
    "Lindo, só que parou de funcionar em uma semana.",
]

entrada = tok(frases, return_tensors="pt", padding=True, truncation=True).to(bert.device)
with torch.no_grad():
    pred = bert(**entrada).logits.argmax(-1).tolist()
for frase, c, b in zip(frases, classico.predict(frases), pred):
    print(f"Clássico: {NOMES[c]:8} | BERT: {NOMES[b]:8} | {frase}")

---
# Parte 2: reconhecer entidades

## 5. Classificar cada token

Na Parte 1, o modelo deu uma categoria ao texto inteiro. No NER, a mesma arquitetura dá uma categoria a **cada token**:
pessoa (PER), organização (ORG), local (LOC), data (DATE) ou nada. O modelo abaixo já vem treinado.

In [ ]:
from transformers import pipeline

ner = pipeline("token-classification", model="Davlan/bert-base-multilingual-cased-ner-hrl",
               aggregation_strategy="first", device=DEVICE)

def entidades(texto):
    return [{"inicio": e["start"], "fim": e["end"], "tipo": e["entity_group"], "confianca": float(e["score"])}
            for e in ner(texto)]

exemplo = "Comprei um notebook da Samsung nas Americanas em São Paulo, e o João, da assistência, resolveu tudo em março."
pd.DataFrame([{"texto": exemplo[e["inicio"]:e["fim"]], "tipo": e["tipo"], "confiança": round(e["confianca"], 2)}
              for e in entidades(exemplo)])

## 6. Palavras-chave em escala

Quais marcas aparecem nas reviews? Sem regra manual, o modelo lê as reviews e devolve as organizações citadas.

In [ ]:
from collections import Counter

textos = reviews.sample(40 if RAPIDO else 300, random_state=0).review_text.str.slice(0, 500).tolist()
marcas = Counter(t[e["inicio"]:e["fim"]] for t in textos for e in entidades(t) if e["tipo"] == "ORG")
pd.DataFrame(marcas.most_common(10), columns=["organização", "menções"])

---
# Parte 3: mascarar dados pessoais

## 7. As conversas do Banco Aurora

Temos 60 mensagens fictícias, com as respostas certas marcadas à mão. Elas têm nomes, locais, empresas, CPF, telefone, e-mail e cartão.

In [ ]:
conversas = [json.loads(linha) for linha in open("dados/conversas_rotuladas.jsonl", encoding="utf-8")]
print(len(conversas), "mensagens")
print(Counter(e["tipo"] for c in conversas for e in c["entidades"]))
print("\n" + conversas[0]["texto"])

## 8. Regras para o que tem formato fixo

CPF, telefone, e-mail e cartão seguem padrões. Uma expressão regular resolve, sem modelo nenhum.

In [ ]:
PADROES = {
    "CARTAO": r"\b\d{4}[ -]\d{4}[ -]\d{4}[ -]\d{4}\b",
    "CPF": r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b",
    "EMAIL": r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b",
    "TEL": r"\(?\b\d{2}\)?[ -]?\d{4,5}-\d{4}\b",
}

def regras(texto):
    return [{"inicio": m.start(), "fim": m.end(), "tipo": tipo, "confianca": 1.0}
            for tipo, padrao in PADROES.items() for m in re.finditer(padrao, texto)]

for frase in ["Meu CPF é 123.456.789-09 e meu telefone é (11) 98765-4321.", "Falei com o Rafael em Campinas."]:
    print(frase, "->", [frase[e["inicio"]:e["fim"]] for e in regras(frase)])

A segunda frase mostra o limite: não existe padrão para "isto é um nome". Para isso precisamos de contexto, e entra o NER.

## 9. Medir por entidade

Em NER, um acerto exige **início, fim e tipo** corretos. Acertar "João" e perder "Pedro Almeida" conta como erro.
A tabela mostra regras e NER trabalhando juntos.

In [ ]:
def prever(texto, limite):
    fixas = regras(texto)
    ocupado = {i for e in fixas for i in range(e["inicio"], e["fim"])}
    return fixas + [e for e in entidades(texto) if e["confianca"] >= limite and e["tipo"] != "DATE"
                    and not ocupado & set(range(e["inicio"], e["fim"]))]

def avaliar(limite):
    certos, previstos, reais = Counter(), Counter(), Counter()
    for c in conversas:
        gabarito = {(e["inicio"], e["fim"], e["tipo"]) for e in c["entidades"]}
        previsao = {(e["inicio"], e["fim"], e["tipo"]) for e in prever(c["texto"], limite)}
        for chave, conjunto in [(certos, gabarito & previsao), (previstos, previsao), (reais, gabarito)]:
            chave.update(t for _, _, t in conjunto)
    linhas = []
    for tipo in sorted(reais):
        p = certos[tipo] / max(previstos[tipo], 1)
        r = certos[tipo] / reais[tipo]
        linhas.append({"tipo": tipo, "precisão": p, "recall": r, "F1": 2 * p * r / max(p + r, 1e-9), "quantas": reais[tipo]})
    return pd.DataFrame(linhas).set_index("tipo").round(2)

avaliar(limite=0.5)

## 10. Decidir o limite de confiança

O modelo dá uma confiança a cada entidade. Um limite baixo mascara mais e erra para o lado da cautela.
Um limite alto mascara menos e deixa mais nomes passarem. Aqui importa outra conta: **quantos nomes saíram do banco?**
Um nome vaza se algum pedaço dele não for coberto por uma máscara.

In [ ]:
def mascarar(texto, previsao):
    for e in sorted(previsao, key=lambda e: e["inicio"], reverse=True):
        texto = texto[:e["inicio"]] + f"[{e['tipo']}]" + texto[e["fim"]:]
    return texto

def vazamentos(limite):
    total, vazou, mascarados_a_mais = 0, 0, 0
    for c in conversas:
        previsao = prever(c["texto"], limite)
        coberto = {i for e in previsao if e["tipo"] in ("PER", "LOC") for i in range(e["inicio"], e["fim"])}
        certo = {i for e in c["entidades"] if e["tipo"] in ("PER", "LOC") for i in range(e["inicio"], e["fim"])}
        for e in c["entidades"]:
            if e["tipo"] in ("PER", "LOC"):
                total += 1
                vazou += not all(i in coberto for i in range(e["inicio"], e["fim"]))
        mascarados_a_mais += sum(1 for e in previsao if e["tipo"] in ("PER", "LOC")
                                 and not any(i in certo for i in range(e["inicio"], e["fim"])))
    return {"limite": limite, "nomes e locais": total, "vazaram": vazou, "máscaras à toa": mascarados_a_mais}

curva = pd.DataFrame([vazamentos(l) for l in [0.0, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]]).set_index("limite")
display(curva)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(curva.index, curva["vazaram"], marker="o", label="vazaram")
ax.plot(curva.index, curva["máscaras à toa"], marker="o", label="máscaras à toa")
ax.set_xlabel("limite de confiança"); ax.set_ylabel("quantidade"); ax.legend(); plt.tight_layout()

Escolha seu limite e veja o que sairia do banco nas mensagens que ainda vazam.

In [ ]:
LIMITE = 0.5

print(vazamentos(LIMITE), "\n")
for c in conversas:
    previsao = prever(c["texto"], LIMITE)
    coberto = {i for e in previsao for i in range(e["inicio"], e["fim"])}
    perdidos = [c["texto"][e["inicio"]:e["fim"]] for e in c["entidades"]
                if e["tipo"] in ("PER", "LOC") and not all(i in coberto for i in range(e["inicio"], e["fim"]))]
    if perdidos:
        print(f"Vazou {perdidos}: {mascarar(c['texto'], previsao)}")

## Para responder

1. Com os resultados da Parte 1, em que situação você ficaria com o modelo clássico?
2. Qual tipo de nome o NER mais perdeu? O que esses casos têm em comum?
3. Qual limite você levaria à diretoria do Banco Aurora, e o que precisaria acontecer com os nomes que ainda vazam?